In [ ]:
# Content based Recommendator

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
movies = pd.read_csv("./tmdb_5000_movies.csv")
credits = pd.read_csv("./tmdb_5000_credits.csv")

In [ ]:
movies = movies.merge(credits, on="title")

In [ ]:
movies.info()

In [ ]:
# Columns to be kept for content based recommendataion:

# id
# title
# overview
# genres
# keywords
# cast
# crew

movies = movies[["movie_id", "title", "overview", "genres", "keywords", "cast", "crew"]]

In [ ]:
movies.head(1)

In [ ]:
movies.isnull().sum()

In [ ]:
movies.dropna(inplace=True)

In [ ]:
movies.isnull().sum()

In [ ]:
movies.duplicated().sum()

In [ ]:
movies["genres"][0]

In [ ]:
import ast


def convert(obj):
    L = []
    for i in ast.literal_eval(obj):
        L.append(i["name"])
    return L

In [ ]:
movies["genres"] = movies["genres"].apply(convert)

In [ ]:
movies.head(1)

In [ ]:
movies["keywords"] = movies["keywords"].apply(convert)

In [ ]:
movies.head(1)

In [ ]:
def convertCast(obj):
    L = []
    counter = 0

    for i in ast.literal_eval(obj):
        if counter != 3:
            L.append(i["name"])
            counter += 1
        else:
            break
    return L

In [ ]:
movies["cast"] = movies["cast"].apply(convertCast)

In [ ]:
movies.head(1)

In [ ]:
def fetchDirector(obj):
    L = []
    for i in ast.literal_eval(obj):
        if i["job"] == "Director":
            L.append(i["name"])
            break
    return L

In [ ]:
movies["crew"] = movies["crew"].apply(fetchDirector)

In [ ]:
movies.head(1)

In [ ]:
movies["overview"][0]

In [ ]:
movies["overview"] = movies["overview"].apply(lambda x: x.split())

In [ ]:
movies["genres"] = movies["genres"].apply(lambda x: [i.replace(" ", "") for i in x])
movies["keywords"] = movies["keywords"].apply(lambda x: [i.replace(" ", "") for i in x])
movies["cast"] = movies["cast"].apply(lambda x: [i.replace(" ", "") for i in x])
movies["crew"] = movies["crew"].apply(lambda x: [i.replace(" ", "") for i in x])

In [ ]:
movies["tags"] = (
    movies["overview"]
    + movies["genres"]
    + movies["keywords"]
    + movies["cast"]
    + movies["crew"]
)

In [ ]:
movies.head()

In [ ]:
new_df = movies[["movie_id", "title", "tags"]]

In [ ]:
new_df["tags"] = new_df["tags"].apply(lambda x: " ".join(x))

In [ ]:
new_df["tags"][0]

In [ ]:
new_df["tags"] = new_df["tags"].apply(lambda x: x.lower())

In [77]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(max_features=5000, stop_words="english")

In [78]:
vectors = cv.fit_transform(new_df["tags"]).toarray()

In [79]:
vectors

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(4806, 5000))

In [ ]:
# Stemming
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()

In [73]:
def stem(text):
    y = []

    for i in text.split():
        y.append(ps.stem(i))

    return " ".join(y)

In [75]:
new_df["tags"] = new_df["tags"].apply(stem)

In [80]:
from sklearn.metrics.pairwise import cosine_similarity

In [82]:
similarity = cosine_similarity(vectors)

In [ ]:
def recommend(movie):
    movie_index = new_df[new_df["title"] == movie].index[0]
    distances = similarity[movie_index]
    movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[
        1:6
    ]

    for i in movies_list:
        print(new_df.iloc[i[0]].title)
    return

In [94]:
recommend("Avatar")

Aliens vs Predator: Requiem
Aliens
Falcon Rising
Independence Day
Titan A.E.
